# 📦 Rosetta — pacote de insumos para o Gemini

Fase intermediária: ainda colamos os 3 arquivos no Gemini do mentor manualmente, mas os 3 saem
daqui, 100% do Databricks, sem digitar nada à mão.

1. **SQL SAP** — DDL da CDS raiz + cadeia de CDS intermediárias (`tab_ddddlsrc`)
2. **Metadados** — `DD03L` + `DD04T` (posição, chave, tipo, tamanho, descrição em PT)
3. **SHOW CREATE TABLE** — estrutura da tabela legada equivalente

Preencha os widgets e rode em ordem. Nada é executado no catálogo — toda consulta passa por
`rosetta.seguranca.sql_leitura` (só `SELECT`/`WITH`/`DESCRIBE`/`SHOW`/`EXPLAIN`).

## 1. 🎛️ Widgets

In [ ]:
dbutils.widgets.removeAll()

dbutils.widgets.text("ddlname", "I_BILLINGDOCUMENT", "01 | DDL Name da CDS View raiz")
dbutils.widgets.text("tabnames_metadados", "", "02 | Tabelas p/ metadados (DD03L), separadas por vírgula — vazio = auto (sql_view_name da cadeia)")
dbutils.widgets.text("tabela_legada", "", "03 | Tabela legada (SHOW CREATE TABLE) — vazio = ddlname em minusculo")
dbutils.widgets.dropdown("seguir_associations", "Nao", ["Sim", "Nao"], "04 | Seguir associations na cadeia")
dbutils.widgets.text("max_profundidade", "15", "05 | Profundidade máxima")
dbutils.widgets.text("idioma_dd04t", "P", "06 | Chave de idioma no DD04T")
dbutils.widgets.dropdown("gravar_arquivos", "Sim", ["Sim", "Nao"], "07 | Gravar em ddl/<DDLNAME>/gemini/")

dbutils.widgets.text("catalog_raw", "platform_dev", "10 | Catalog origem")
dbutils.widgets.text("schema_raw", "sap_s4_nc2_raw", "11 | Schema origem (raw)")
dbutils.widgets.text("tabela_ddl", "tab_ddddlsrc", "12 | Tabela DDDDLSRC")
dbutils.widgets.text("tabela_dep", "ddldependency", "13 | Tabela DDLDEPENDENCY")
dbutils.widgets.text("tabela_dd04t", "dd04t", "14 | Tabela DD04T (schema raw)")
dbutils.widgets.text("catalog_target", "platform_dev", "15 | Catalog destino")
dbutils.widgets.text("schema_target", "sap_s4_nc2_replica", "16 | Schema destino")
dbutils.widgets.text("tabela_dd03l", "dd03l", "17 | Tabela DD03L (schema destino)")
dbutils.widgets.text("catalog_legado", "platform_dev", "18 | Catalog da tabela legada (SHOW CREATE TABLE)")
dbutils.widgets.text("schema_legado", "sap_s4_nc2_raw", "19 | Schema da tabela legada — ajuste por ambiente (ex.: platform/sap_s4_replica em prod)")

print("🎛️  Widgets criados.")

## 2. 📦 Carregar o pacote `rosetta`

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
from pathlib import Path

_aqui = Path.cwd()
for _cand in [_aqui, *_aqui.parents]:
    if (_cand / "src" / "rosetta" / "__init__.py").exists():
        RAIZ = _cand
        break
else:
    raise FileNotFoundError(f"Raiz do repositório não encontrada a partir de {_aqui}")

if str(RAIZ / "src") not in sys.path:
    sys.path.insert(0, str(RAIZ / "src"))

import rosetta
from rosetta import Config, Contexto
from rosetta.insumos_gemini import montar_pacote, salvar_pacote

PASTA_DDL = RAIZ / "ddl"

print(f"📦 rosetta {rosetta.__version__}")
print(f"📁 raiz do repo : {RAIZ}")

## 3. 📖 Configuração e índice de fontes

In [ ]:
def _w(nome, padrao=""):
    try:
        return (dbutils.widgets.get(nome) or "").strip() or padrao
    except Exception:
        return padrao

DDLNAME          = _w("ddlname").upper()
SEGUIR_ASSOC     = _w("seguir_associations", "Nao") == "Sim"
MAX_PROFUNDIDADE = int(_w("max_profundidade", "15"))
IDIOMA           = _w("idioma_dd04t", "P")
GRAVAR           = _w("gravar_arquivos", "Sim") == "Sim"

_tabnames_raw = _w("tabnames_metadados", "")
# None = deixa montar_pacote derivar automaticamente via sql_view_name da cadeia
# (o DD03L não indexa pelo nome da CDS, e sim pela estrutura/SQL view gerada).
TABNAMES_META = [t.strip() for t in _tabnames_raw.split(",") if t.strip()] or None
TABELA_LEGADA = _w("tabela_legada", "") or DDLNAME.lower()

CFG = Config.de_widgets(dbutils)
print(CFG.resumo())
print(f"\n🎯 DDL raiz: '{DDLNAME}'")
print(f"📊 Tabelas p/ metadados (DD03L): {TABNAMES_META or 'auto (derivado da cadeia após montar o pacote)'}")
print(f"🏚️  Tabela legada (SHOW CREATE TABLE): {CFG.fqn_legado}.{TABELA_LEGADA}")

In [ ]:
if "CTX" not in globals() or CTX.cfg != CFG:
    CTX = Contexto(spark, CFG, raiz_ddl=PASTA_DDL)
else:
    print(f"📇 Índice já em memória: {len(CTX.indice):,} entradas")

## 4. 🛠️ Montar o pacote (SQL SAP + metadados + SHOW CREATE TABLE)

In [ ]:
if not CTX.existe(DDLNAME):
    print(f"❓ '{DDLNAME}' não existe em {CFG.fqn_ddl}.")
    PACOTE = None
else:
    PACOTE = montar_pacote(
        spark, CFG, CTX.indice, CTX.resolvedor,
        ddlname=DDLNAME,
        tabela_legada=TABELA_LEGADA,
        tabnames_metadados=TABNAMES_META,
        seguir_assoc=SEGUIR_ASSOC,
        max_profundidade=MAX_PROFUNDIDADE,
        idioma=IDIOMA,
    )
    print(f"✅ Cadeia CDS: {', '.join(PACOTE.cadeia.ddlnames)}")
    print(f"✅ Tabelas usadas no DD03L (auto ou manual): {PACOTE.tabnames_metadados}")
    print(f"✅ Linhas de metadados (DD03L+DD04T): {len(PACOTE.metadados_linhas)}")
    if not PACOTE.metadados_linhas:
        print("   ⚠️  Vazio — confira se esses tabnames existem em "
              f"{CFG.fqn_dd03l} (rode SELECT DISTINCT tabname FROM {CFG.fqn_dd03l} "
              "WHERE tabname IN (...) pra conferir).")
    print(f"✅ SHOW CREATE TABLE: {'ok' if PACOTE.show_create_table else 'vazio'}")

## 5. 📄 Arquivo 1 — SQL SAP

In [ ]:
if PACOTE is not None:
    print(PACOTE.arquivo_1_sql_sap)

## 6. 📄 Arquivo 2 — Metadados (DD03L + DD04T)

In [ ]:
if PACOTE is not None:
    print(PACOTE.arquivo_2_metadados)

## 7. 📄 Arquivo 3 — SHOW CREATE TABLE

In [ ]:
if PACOTE is not None:
    print(PACOTE.arquivo_3_show_create_table)

## 8. 💾 Salvar em `ddl/<DDLNAME>/gemini/`

In [ ]:
if PACOTE is not None:
    caminhos = salvar_pacote(PASTA_DDL, PACOTE, gravar=GRAVAR)
    cabec = "💾 Gravado em" if GRAVAR else "🔍 Simulação (nada gravado) —"
    print(f"{cabec} {PASTA_DDL / rosetta.nome_pasta(DDLNAME) / 'gemini'}")
    for rotulo, caminho in caminhos.items():
        print(f"   • {rotulo}: {caminho.name}")
    print("\n👉 Copie os 3 arquivos para o prompt do Gemini (referencias/Texto gems.txt).")